# Test OpenRouter API Key
Notebook này được dùng để kiểm tra xem OpenRouter API Key của bạn có hợp lệ và hoạt động bình thường hay không.

In [1]:
import os
from dotenv import load_dotenv
import requests
import json

# Tải biến môi trường từ file .env ở thư mục gốc
load_dotenv(dotenv_path='../.env')

# Lấy API key từ biến môi trường hoặc gán cứng vào đây:
api_key = os.getenv('OPENROUTER_API_KEY', '')

if not api_key:
    print("⚠️ CẢNH BÁO: Chưa tìm thấy OPENROUTER_API_KEY. Vui lòng thiết lập trong file .env hoặc gán trực tiếp vào code.")
else:
    print("✅ Đã tìm thấy API Key bắt đầu bằng:", api_key + "...")

In [2]:
# 1. Gửi request test cơ bản bằng requests (test mạng và API key)
url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
data = {
    # Sử dụng một model miễn phí để test kết nối
    "model": "nvidia/nemotron-3-super-120b-a12b:free", 
    "messages": [
        {"role": "user", "content": "Xin chào, bạn có nghe rõ không?"}
    ]
}

if api_key:
    print("Đang gửi request tới OpenRouter...")
    try:
        response = requests.post(url, headers=headers, json=data)
        if response.status_code == 200:
            print("\n🎉 TEST THÀNH CÔNG!")
            result = response.json()
            print("Phản hồi từ LLM:", result['choices'][0]['message']['content'])
        else:
            print(f"\n❌ TEST THẤT BẠI. Mã lỗi: {response.status_code}")
            print("Chi tiết lỗi:", response.text)
    except Exception as e:
        print(f"\n❌ Gặp lỗi trong lúc gửi request: {e}")

In [5]:
# 2. Test thông qua class OpenRouterClient nội bộ của dự án
import sys
sys.path.append('..')

if api_key:
    try:
        from src.openrouter_client import OpenRouterClient
        print("Đang test thông qua OpenRouterClient của dự án...")
        
        # Tạo client với model mặc định hoặc model miễn phí để test
        client = OpenRouterClient(api_key=api_key, model="google/gemma-4-26b-a4b-it:free")
        
        response_text = client.complete(
            system_prompt="Bạn là một trợ lý ảo.",
            user_prompt="Hãy nói 'Chào bạn' ngắn gọn.",
            max_tokens=50
        )
        print("\n✅ OpenRouterClient hoạt động tốt!")
        print("Phản hồi:", response_text)
    except Exception as e:
        print("\n❌ OpenRouterClient gặp lỗi:")
        print(str(e))